# IFEST DAC 2026 — V15 BASELINE (official-only, pretrained-free)
Sparse TF-IDF LogReg + factual HGB + meta-LogReg. Tanpa pretrained/LLM/API, tanpa data
eksternal, tanpa pseudo-label test (patuh `docs/rules.md`).

Isi: evaluasi nested BERSIH (outer 5-fold, inner 2-fold pooled) lalu fitting final +
submission. Baseline evaluasi kerja — BUKAN klaim mengalahkan V12A / top-5.
Artefak: submission.csv, thresholds.json, manifest.json, oof + metrik per fold.


In [2]:
import csv, re, unicodedata, math, json, time, random
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)

def find_competition_csv(name):
    cands = [Path("data") / name, Path.cwd() / "data" / name]
    try:
        kg = Path("/kaggle/input")
        if kg.exists():
            cands += sorted(kg.rglob(name))
    except Exception:
        pass
    for p in cands:
        if p.exists():
            return p
    raise FileNotFoundError(name)

TRAIN_P = find_competition_csv("train.csv")
TEST_P = find_competition_csv("test.csv")
SMP_P = find_competition_csv("sample_submission.csv")
try:
    OUT = Path("/kaggle/working/results_v15_nb"); OUT.mkdir(parents=True, exist_ok=True)
except Exception:
    OUT = Path("results/v15_notebook"); OUT.mkdir(parents=True, exist_ok=True)
print("train:", TRAIN_P, "test:", TEST_P, "sample:", SMP_P, "out:", OUT, flush=True)

rows = []
with open(TRAIN_P, encoding="utf-8") as f:
    for r in csv.DictReader(f):
        rows.append([r["id"], r["title"] or "", r["content"] or "", int(r["label"])])
y = np.array([r[3] for r in rows])
assert len(rows) == 14397, len(rows)
c = Counter(y.tolist())
print("n_train=14397 ok, label balance:", dict(c), "neg_rate=%.4f" % (c[0] / len(y)), flush=True)


train: data/train.csv test: data/test.csv sample: data/sample_submission.csv out: results/v15_notebook
n_train=14397 ok, label balance: {0: 1437, 1: 12960} neg_rate=0.0998



In [3]:
def norm(s):
    s = unicodedata.normalize("NFKC", s or "").lower()
    return re.sub(r"\s+", " ", s).strip()
def toks(s):
    return re.findall(r"[a-z0-9]+", norm(s))

lt = np.array([len(t) for _, t, _, _ in rows]); lc = np.array([len(x) for _, _, x, _ in rows])
print("judul char p50=%d p90=%d | konten char p50=%d p90=%d" % (
    int(np.median(lt)), int(np.percentile(lt, 90)), int(np.median(lc)), int(np.percentile(lc, 90))), flush=True)

parent = list(range(len(rows)))
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra
mp_t, mp_c = defaultdict(list), defaultdict(list)
for i, (_, t, x, _) in enumerate(rows):
    mp_t[norm(t)].append(i); mp_c[norm(x)].append(i)
for v in list(mp_t.values()) + list(mp_c.values()):
    for j in v[1:]: union(v[0], j)
groups = np.array([find(i) for i in range(len(rows))])
print("grup DSU (exact judul/body): %d utk %d baris" % (len(set(groups.tolist())), len(rows)), flush=True)


judul char p50=62 p90=76 | konten char p50=1893 p90=3210
grup DSU (exact judul/body): 11526 utk 14397 baris



In [4]:
NEG = {"tidak", "tak", "bukan", "jangan", "belum", "tanpa", "batal", "gagal", "hoaks", "hoax", "bohong", "palsu", "keliru"}
def build_idf(docs):
    df = Counter()
    for d in docs:
        for w in set(d): df[w] += 1
    n = len(docs)
    return {w: math.log((n - df[w] + 0.5) / (df[w] + 0.5) + 1.0) for w in df}, float(np.mean([len(d) for d in docs]))
def bm25(qt, dt, idf, avgdl, k1=1.5, b=0.75):
    tf = Counter(dt); dl = len(dt); s = 0.0
    for w in qt:
        if w not in idf: continue
        f = tf.get(w, 0)
        if f: s += idf[w] * f * (k1 + 1) / (f + k1 * (1 - b + b * dl / max(avgdl, 1)))
    return s
def caps_phrases(s):
    return re.findall(r"\b[A-Z][A-Za-z]{2,}(?:\s+[A-Z][A-Za-z]{2,})?", s or "")
def num_units(t):
    return [m.group(0).strip() for m in re.finditer(r"\d+(?:[.,]\d+)?\s*[a-z%]*", norm(t))]
def factual(t, x, idf, avgdl):
    tt, ct = toks(t), toks(x); st, sc = set(tt), set(ct)
    cov = len(st & sc) / max(len(st), 1)
    tb = set(zip(tt, tt[1:])); cb = set(zip(ct, ct[1:]))
    bcov = len(tb & cb) / max(len(tb), 1)
    best = 0.0; W = 30
    if ct and st:
        for i in range(0, max(len(ct) - W + 1, 1), 10):
            win = set(ct[i:i + W]); v = len(st & win) / max(len(st), 1)
            if v > best: best = v
    bm = bm25(tt, ct, idf, avgdl) / max(len(tt), 1)
    nu = num_units(t); lcx = norm(x)
    num_sup = sum(1 for u in nu if u in lcx) / max(len(nu), 1) if nu else 1.0
    nt = set(re.findall(r"\d+", t or "")); nc = set(re.findall(r"\d+", x or ""))
    num_miss = 0.0 if not nt or (nt & nc) else 1.0
    cp = caps_phrases(t); low = set(re.findall(r"[a-z0-9]+", norm(x)))
    caps = sum(1 for p in cp if all(w in low for w in toks(p))) / max(len(cp), 1) if cp else 1.0
    gt = 1.0 if set(tt) & NEG else 0.0; gc = 1.0 if set(ct) & NEG else 0.0
    return [cov, bcov, best, bm, num_sup, num_miss, caps, gt, gc,
            1.0 if gt != gc else 0.0, len(tt), len(ct), len(tt) / max(len(ct), 1)]

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from scipy.sparse import hstack

THR = np.arange(0.2, 0.81, 0.02)
def pick_thr(yv, s):
    bt, bf = 0.5, -1.0
    for t in THR:
        f = f1_score(yv, (s >= t).astype(int), average="macro", zero_division=0)
        if f > bf: bf, bt = f, float(t)
    return bt, float(bf)
def titles(i): return rows[i][1]
def pair_txt(i): return rows[i][1] + " [SEP] " + rows[i][2][:512]
def fit_sparse(txt_tr, ytr):
    vw = TfidfVectorizer(ngram_range=(1, 2), max_features=20000, sublinear_tf=True, min_df=2)
    vc = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=15000, sublinear_tf=True, min_df=2)
    Xtr = hstack([vw.fit_transform(txt_tr), vc.fit_transform(txt_tr)])
    m = LogisticRegression(max_iter=2000, class_weight="balanced", C=0.5).fit(Xtr, ytr)
    return (vw, vc, m)
def pred_sparse(bundle, txt):
    vw, vc, m = bundle
    return m.predict_proba(hstack([vw.transform(txt), vc.transform(txt)]))[:, 1]
def fit_factual(pairs_tr, ytr):
    docs = [toks(t) + toks(x) for t, x in pairs_tr]
    idf, avgdl = build_idf(docs)
    F = np.array([factual(t, x, idf, avgdl) for t, x in pairs_tr])
    sc = StandardScaler().fit(F)
    mh = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                        l2_regularization=2.0, class_weight="balanced", random_state=SEED)
    mh.fit(sc.transform(F), ytr)
    return (idf, avgdl, sc, mh)
def pred_factual(bundle, pairs):
    idf, avgdl, sc, mh = bundle
    F = np.array([factual(t, x, idf, avgdl) for t, x in pairs])
    return mh.predict_proba(sc.transform(F))[:, 1]
print("builders ok (sparse lead512-pair, factual FULL-content)", flush=True)


builders ok (sparse lead512-pair, factual FULL-content)



In [5]:
# Evaluasi nested: outer 5-fold; inner 2-fold pooled utk meta+threshold; nilai di outer-val.
# Dilaporkan mean-fold DAN pooled-OOF (keduanya benar, beda penamaan).
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
oof = {k: np.zeros(len(y)) for k in ("s_fa", "s_st")}
oof_fold = np.full(len(y), -1); oof_pfa = np.zeros(len(y)); oof_pst = np.zeros(len(y))
fa_f, st_f, tfa_f, tst_f = [], [], [], []
t0 = time.time()
for fi, (tro, vao) in enumerate(outer.split(np.zeros(len(y)), y, groups)):
    gtro = groups[tro]
    pos = {idx: k for k, idx in enumerate(tro.tolist())}
    in_sp = np.zeros(len(tro)); in_fa = np.zeros(len(tro))
    for a, b in StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=100 + fi).split(np.zeros(len(tro)), y[tro], gtro):
        itr, iva = tro[a], tro[b]
        sp_b = fit_sparse([pair_txt(i) for i in itr], y[itr])
        fa_b = fit_factual([(titles(i), rows[i][2]) for i in itr], y[itr])
        li = [pos[i] for i in iva.tolist()]
        in_sp[li] = pred_sparse(sp_b, [pair_txt(i) for i in iva])
        in_fa[li] = pred_factual(fa_b, [(titles(i), rows[i][2]) for i in iva])
    thr_fa, _ = pick_thr(y[tro], in_fa)
    meta = LogisticRegression(max_iter=1000, class_weight="balanced")
    meta.fit(np.column_stack([in_sp, in_fa]), y[tro])
    thr_st, _ = pick_thr(y[tro], meta.predict_proba(np.column_stack([in_sp, in_fa]))[:, 1])
    sp_F = fit_sparse([pair_txt(i) for i in tro], y[tro])
    fa_F = fit_factual([(titles(i), rows[i][2]) for i in tro], y[tro])
    s_sp = pred_sparse(sp_F, [pair_txt(i) for i in vao])
    s_fa = pred_factual(fa_F, [(titles(i), rows[i][2]) for i in vao])
    s_st = meta.predict_proba(np.column_stack([s_sp, s_fa]))[:, 1]
    oof["s_fa"][vao] = s_fa; oof["s_st"][vao] = s_st; oof_fold[vao] = fi
    oof_pfa[vao] = (s_fa >= thr_fa).astype(int); oof_pst[vao] = (s_st >= thr_st).astype(int)
    tfa_f.append(thr_fa); tst_f.append(thr_st)
    fa_f.append(float(f1_score(y[vao], oof_pfa[vao], average="macro", zero_division=0)))
    st_f.append(float(f1_score(y[vao], oof_pst[vao], average="macro", zero_division=0)))
    print("fold %d fa=%.4f(t=%.2f) st=%.4f(t=%.2f) (%ds)" % (fi, fa_f[-1], thr_fa, st_f[-1], thr_st, time.time() - t0), flush=True)
pooled_fa = float(f1_score(y, oof_pfa, average="macro", zero_division=0))
pooled_st = float(f1_score(y, oof_pst, average="macro", zero_division=0))
print("mean-fold factual=%.4f stack=%.4f" % (float(np.mean(fa_f)), float(np.mean(st_f))), flush=True)
print("pooled-OOF factual=%.4f stack=%.4f (prediksi per-fold dgn threshold beku-nya)" % (pooled_fa, pooled_st), flush=True)
with open(OUT / "metrics_nested.json", "w", encoding="utf-8") as f:
    json.dump({"fa_folds": fa_f, "st_folds": st_f, "thr_fa": tfa_f, "thr_st": tst_f,
               "mean_fa": float(np.mean(fa_f)), "mean_st": float(np.mean(st_f)),
               "pooled_fa": pooled_fa, "pooled_st": pooled_st}, f, indent=2)
with open(OUT / "oof_nested.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f); w.writerow(["id", "y", "fold", "s_fa", "s_st", "pred_fa", "pred_st"])
    for i, (rid, _, _, yy) in enumerate(rows):
        w.writerow([rid, yy, int(oof_fold[i]), "%.6f" % oof["s_fa"][i], "%.6f" % oof["s_st"][i],
                    int(oof_pfa[i]), int(oof_pst[i])])
print("nested artifacts saved", flush=True)


fold 0 fa=0.6066(t=0.34) st=0.6265(t=0.34) (41s)
fold 1 fa=0.5853(t=0.34) st=0.5758(t=0.34) (82s)
fold 2 fa=0.5754(t=0.32) st=0.5953(t=0.32) (122s)
fold 3 fa=0.5907(t=0.38) st=0.5967(t=0.36) (164s)
fold 4 fa=0.5866(t=0.38) st=0.5834(t=0.36) (206s)
mean-fold factual=0.5889 stack=0.5955
pooled-OOF factual=0.5897 stack=0.5952 (prediksi per-fold dgn threshold beku-nya)
nested artifacts saved



In [6]:
# Fitting FINAL utk submission (terpisah dari evaluasi di atas):
# 5-fold cross-fit skor dasar -> meta + median threshold -> refit dasar di FULL train -> inferensi test.
# Threshold submission = median threshold per-fold (heuristik standar); angka performa tetap milik nested.
cf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
S_sp = np.zeros(len(y)); S_fa = np.zeros(len(y))
thrs_fa, thrs_st = [], []
for tro, vao in cf.split(np.zeros(len(y)), y, groups):
    sp_b = fit_sparse([pair_txt(i) for i in tro], y[tro])
    fa_b = fit_factual([(titles(i), rows[i][2]) for i in tro], y[tro])
    S_sp[vao] = pred_sparse(sp_b, [pair_txt(i) for i in vao])
    S_fa[vao] = pred_factual(fa_b, [(titles(i), rows[i][2]) for i in vao])
metaF = LogisticRegression(max_iter=1000, class_weight="balanced")
metaF.fit(np.column_stack([S_sp, S_fa]), y)
S_st = metaF.predict_proba(np.column_stack([S_sp, S_fa]))[:, 1]
for _, vao in StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y, groups):
    tf, _ = pick_thr(y[vao], S_fa[vao]); ts, _ = pick_thr(y[vao], S_st[vao])
    thrs_fa.append(tf); thrs_st.append(ts)
THR_FA, THR_ST = float(np.median(thrs_fa)), float(np.median(thrs_st))
print("final median thresholds: factual=%.2f stack=%.2f" % (THR_FA, THR_ST), flush=True)

sp_full = fit_sparse([pair_txt(i) for i in range(len(rows))], y)
fa_full = fit_factual([(titles(i), rows[i][2]) for i in range(len(rows))], y)
test_rows = []
with open(TEST_P, encoding="utf-8") as f:
    for r in csv.DictReader(f):
        test_rows.append([r["id"], r["title"] or "", r["content"] or ""])
assert len(test_rows) == 3603, len(test_rows)
tte = [t + " [SEP] " + x[:512] for _, t, x in test_rows]
ste_sp = pred_sparse(sp_full, tte)
ste_fa = pred_factual(fa_full, [(t, x) for _, t, x in test_rows])
ste = metaF.predict_proba(np.column_stack([ste_sp, ste_fa]))[:, 1]
pred = (ste >= THR_ST).astype(int)
with open(OUT / "submission.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f); w.writerow(["id", "label"])
    for (rid, _, _), p in zip(test_rows, pred):
        w.writerow([rid, int(p)])
with open(OUT / "thresholds.json", "w", encoding="utf-8") as f:
    json.dump({"thr_factual_median": THR_FA, "thr_stack_median": THR_ST,
               "thr_factual_folds": thrs_fa, "thr_stack_folds": thrs_st}, f, indent=2)
with open(OUT / "manifest.json", "w", encoding="utf-8") as f:
    json.dump({"seed": SEED, "n_train": len(rows), "n_test": len(test_rows),
               "groups": int(len(set(groups.tolist()))),
               "towers": "sparse-C0.5-lead512pair + factualHGB-13-full + metaLogReg",
               "external": "none", "pretrained": "none",
               "rules": "official-only, test-sealed, no-pretrained"}, f, indent=2)
print("submission + thresholds + manifest saved", flush=True)


final median thresholds: factual=0.36 stack=0.34
submission + thresholds + manifest saved



In [7]:
smp = list(csv.DictReader(open(SMP_P, encoding="utf-8")))
sub = list(csv.DictReader(open(OUT / "submission.csv", encoding="utf-8")))
assert len(sub) == 3603 and len(smp) == 3603
assert [r["id"] for r in sub] == [r["id"] for r in smp]
assert set(r["label"] for r in sub) <= {"0", "1"}
print("submission VALID: 3603 baris, id sesuai sample, label biner:", dict(Counter(r["label"] for r in sub)), flush=True)
print("VERDICT: baseline evaluasi kerja (nested mean-fold stack ~0.59x). Bukan bukti vs V12A / top-5.", flush=True)


submission VALID: 3603 baris, id sesuai sample, label biner: {'1': 3178, '0': 425}
VERDICT: baseline evaluasi kerja (nested mean-fold stack ~0.59x). Bukan bukti vs V12A / top-5.

